In [ ]:
import requests
import pandas as pd
from datetime import datetime

plant_id = input("Enter plant id: ")

plant_master = pd.read_csv(r"D:\Portfolio Github\Pravaah\data\plant_master.csv")
plant_info = plant_master[plant_master['plant_id'] == plant_id]
plant_info = plant_info[['plant_id', 'plant_type', 'capacity_mw', 'latitude', 'longitude']]

# Get current timestamp automatically
timestamp = datetime.now()
print("Current timestamp:", timestamp)

# Assuming plant location from plant_info (e.g., lat/lon); adjust as needed
lat = plant_info['latitude'].iloc[0] 
lon = plant_info['longitude'].iloc[0] 

# API endpoint for 24-hour forecast (hourly data)
url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&hourly=temperature_2m,cloudcover,windspeed_10m,winddirection_10m,shortwave_radiation&forecast_days=2&timezone=auto"

response = requests.get(url)
data = response.json()

# Convert to DataFrame
weather = pd.DataFrame(data['hourly'])

# Rename columns (clean names)
weather.rename(columns={
    'time': 'timestamp',
    'temperature_2m': 'temperature',
    'cloudcover': 'cloud_cover',
    'windspeed_10m': 'wind_speed',
    'winddirection_10m': 'wind_direction',
    'shortwave_radiation': 'irradiance'
}, inplace=True)

# Convert timestamp column to datetime
weather['timestamp'] = pd.to_datetime(weather['timestamp'])

# (Optional) Filter data starting from current time
weather = weather[weather['timestamp'] >= timestamp]

# Merge weather and plant_info by adding plant_info columns to weather (since plant_info has one row for the plant)
info = weather.assign(**plant_info.iloc[0].to_dict())

info.drop(columns=['latitude', 'longitude'], inplace=True)

In [ ]:
import os

# Define the base folder path for irradiance models
model_folder = r"D:\Portfolio Github\Pravaah\models\irradiance"

# List all files and filter for the plant_id
model_files = [f for f in os.listdir(model_folder) if plant_id in f and f.endswith('.pkl')]

if model_files:
    model_path = os.path.join(model_folder, model_files[0])
    print(f"Found model: {model_path}")
else:
    print(f"No model found for {plant_id} in {model_folder}")


In [9]:
from src.irradiance import predict_irradiance
df = predict_irradiance("D:\Portfolio Github\Pravaah\models\irradiance\PLANT_001_irradiance_20260505_1348.pkl", info['timestamp'])

<>:2: SyntaxWarning: invalid escape sequence '\P'
<>:2: SyntaxWarning: invalid escape sequence '\P'
C:\Users\SHUCHISMITA MALLICK\AppData\Local\Temp\ipykernel_23316\1661434967.py:2: SyntaxWarning: invalid escape sequence '\P'
  df = predict_irradiance("D:\Portfolio Github\Pravaah\models\irradiance\PLANT_001_irradiance_20260505_1348.pkl", info['timestamp'])


In [10]:
df

,plant_id,timestamp,irradiance_forecast,lower_90,upper_90,method
0,PLANT_001,2026-05-05 19:00:00,0.00,0.00,0.00,blend_prophet_physics
1,PLANT_001,2026-05-05 20:00:00,0.00,0.00,0.00,blend_prophet_physics
2,PLANT_001,2026-05-05 21:00:00,0.00,0.00,0.00,blend_prophet_physics
3,PLANT_001,2026-05-05 22:00:00,0.00,0.00,0.00,blend_prophet_physics
4,PLANT_001,2026-05-05 23:00:00,0.00,0.00,0.00,blend_prophet_physics
5,PLANT_001,2026-05-06 00:00:00,0.00,0.00,0.00,blend_prophet_physics
6,PLANT_001,2026-05-06 01:00:00,0.00,0.00,0.00,blend_prophet_physics
7,PLANT_001,2026-05-06 02:00:00,0.00,0.00,0.00,blend_prophet_physics
8,PLANT_001,2026-05-06 03:00:00,0.00,0.00,0.00,blend_prophet_physics
9,PLANT_001,2026-05-06 04:00:00,0.00,0.00,0.00,blend_prophet_physics


In [11]:
from src.health_factor import predict_health_factor
df = predict_health_factor("D:\Portfolio Github\Pravaah\models\health_factor\PLANT_001_health_20260505_1808.pkl", info['timestamp'])
df.head()

,plant_id,timestamp,health_forecast,lower_90,upper_90,repair_probability,method
0,PLANT_001,2026-05-05 19:00:00,0.6689,0.6674,0.6704,0.0003,linear_trend
1,PLANT_001,2026-05-05 20:00:00,0.6689,0.6674,0.6704,0.0003,linear_trend
2,PLANT_001,2026-05-05 21:00:00,0.6689,0.6674,0.6703,0.0003,linear_trend
3,PLANT_001,2026-05-05 22:00:00,0.6688,0.6674,0.6703,0.0003,linear_trend
4,PLANT_001,2026-05-05 23:00:00,0.6688,0.6674,0.6703,0.0003,linear_trend


In [12]:
from src.availability import predict_availability
df = predict_availability(r"D:\Portfolio Github\Pravaah\models\availability\PLANT_001_availability_20260505_1939.pkl", info['timestamp'])
df.head()

,plant_id,timestamp,availability_forecast,lower_90,upper_90,method
0,PLANT_001,2026-05-05 19:00:00,81.83,73.09,73.28,blend_prophet_baseline
1,PLANT_001,2026-05-05 20:00:00,81.83,73.09,73.28,blend_prophet_baseline
2,PLANT_001,2026-05-05 21:00:00,81.83,73.08,73.28,blend_prophet_baseline
3,PLANT_001,2026-05-05 22:00:00,81.83,73.09,73.28,blend_prophet_baseline
4,PLANT_001,2026-05-05 23:00:00,81.83,73.09,73.29,blend_prophet_baseline


In [ ]:
irradiance_model_path = models["irradiance"]['model']
irradiance_model = joblib.load(irradiance_model_path)

irradiance_pred = irradiance_model.predict(features)

FileNotFoundError: [Errno 2] No such file or directory: 'prophet'

In [ ]:
irradiance_pred = irradiance_model.predict(info[['timestamp', 'plant_id']])

irradiance_pred

AttributeError: 'str' object has no attribute 'predict'

In [ ]:
print(models["irradiance"].keys())

dict_keys(['model', 'plant_id', 'plant_meta', 'horizon', 'model_name'])
